# Sharp Corner Microchannel (3D, XNSE-Solver)

In [ ]:
#r "BoSSSpad/BoSSSpad.dll"
using System;
using System.Collections.Generic;
using System.Linq;
using ilPSP;
using ilPSP.Utils;
using BoSSS.Platform;
using BoSSS.Foundation;
using BoSSS.Foundation.XDG;
using BoSSS.Foundation.Grid;
using BoSSS.Foundation.Grid.Classic;
using BoSSS.Foundation.IO;
using BoSSS.Solution;
using BoSSS.Solution.Control;
using BoSSS.Solution.GridImport;
using BoSSS.Solution.Statistic;
using BoSSS.Solution.Utils;
using BoSSS.Solution.AdvancedSolvers;
using BoSSS.Solution.Gnuplot;
using BoSSS.Application.BoSSSpad;
using BoSSS.Application.XNSE_Solver;
using static BoSSS.Application.BoSSSpad.BoSSSshell;
Init();

In [ ]:
using BoSSS.Solution.NSECommon;
using BoSSS.Solution.XNSECommon;
using BoSSS.Solution.LevelSetTools;
using BoSSS.Solution.LevelSetTools.SolverWithLevelSetUpdater;
using BoSSS.Solution.XdgTimestepping;

In [ ]:
var myBatch = ExecutionQueues[1];
myBatch

RuntimeLocation,AdditionalEnvironmentVars,DeploymentBaseDirectory,DeployRuntime,Name,DotnetRuntime,Username,NumOfAdditionalServiceCores,NumOfAdditionalServiceCoresMPISerial,NumOfServiceCoresPerMPIprocess,ServerName,ComputeNodes,DefaultJobPriority,SingleNode,AllowedDatabasesPaths
win\amd64,[ ],\\dc3\userspace\smuda\hpccluster\binaries,True,FDY-WindowsHPC,dotnet,FDY\smuda,0,0,0,DC3,<null>,Normal,True,[ \\dc3\userspace\smuda\hpccluster\ ]


In [ ]:
BoSSSshell.WorkflowMgm.Init("SharpCornerMicroChannel3D", myBatch);

Project name is set to 'SharpCornerMicroChannel3D'.
Opening existing database '\\dc3\userspace\smuda\hpccluster\SharpCornerMicroChannel3D'.


In [ ]:
wmg.Sessions

#0: SharpCornerMicroChannel3D	SCMC3D_nSec3_grdRes2_FlowRate420_dt0.0001_PARDISO	03/14/2025 14:09:37	73f610b6...
#1: SharpCornerMicroChannel3D	SCMC3D_nSec3_grdRes2_FlowRate510*	03/19/2025 15:42:16	63a625fb...
#2: SharpCornerMicroChannel3D	SCMC3D_nSec3_grdRes2_FlowRate540_dt0.0001_PARDISO*	03/19/2025 15:41:29	9367a042...
#3: SharpCornerMicroChannel3D	SCMC3D_nSec3_grdRes2_FlowRate480	03/14/2025 09:19:34	14b48b4d...
#4: SharpCornerMicroChannel3D	SCMC3D_nSec3_grdRes2_FlowRate420_dt0.0001*	03/14/2025 13:34:00	70a66743...
#5: SharpCornerMicroChannel3D	SCMC3D_nSec3_grdRes2_FlowRate300	03/13/2025 14:28:07	f0fe3305...
#6: SharpCornerMicroChannel3D	SCMC3D_nSec3_grdRes2_FlowRate420	03/13/2025 14:29:22	1e9612b0...
#7: SharpCornerMicroChannel3D	SCMC3D_nSec3_grdRes2_FlowRate660*	03/13/2025 14:31:50	5dc30c78...
#8: SharpCornerMicroChannel3D	SCMC3D_nSec3_grdRes2_FlowRate540*	03/13/2025 14:31:49	fc974fc6...
#9: SharpCornerMicroChannel3D	SCMC3D_nSec3_grdRes2	01/29/2025 14:37:14	26b02f67...
#10: SharpCorn

In [ ]:
//wmg.Sessions.Pick(0).Delete(true)

In [ ]:
var sess = wmg.Sessions.Pick(0);
sess

SharpCornerMicroChannel3D	SCMC3D_nSec3_grdRes2_FlowRate420_dt0.0001_PARDISO	03/14/2025 14:09:37	73f610b6...

In [ ]:
//sess.Export().WithSupersampling(3).Do();

## Evaluate steady state for transient simulations

In [ ]:
var timesteps = sess.Timesteps.Skip(2);
int numTs = timesteps.Count();
numTs

101

In [ ]:
string[] fieldNames = new string[] { "VelocityX", "VelocityY", "VelocityZ", "Pressure" };
double[][] absFieldDiffL2Norm = new double[fieldNames.Length][];  
for (int fn = 0; fn < fieldNames.Length; fn++) {
    absFieldDiffL2Norm[fn] = new double[numTs-1];
}
double[] time = new double[numTs-1];

for (int ts = 1; ts < numTs; ts++) {
    time[ts-1] = timesteps.Pick(ts).PhysicalTime;

    for (int fn = 0; fn < fieldNames.Length; fn++) {

        var currStepField = timesteps.Pick(ts).GetField(fieldNames[fn]);
        var prevStepField = timesteps.Pick(ts-1).GetField(fieldNames[fn]);
        double err = currStepField.L2Error(prevStepField);

        absFieldDiffL2Norm[fn][ts-1] = err;
    }
}

In [ ]:
Plot2Ddata plotFieldDiff_Vel = new Plot2Ddata();
plotFieldDiff_Vel.Title = "Velocity field diff errors over time";
plotFieldDiff_Vel.Xlabel = "time t";
plotFieldDiff_Vel.Ylabel = "diff field L2 error norm";

Plot2Ddata plotFieldDiff_P = new Plot2Ddata();
plotFieldDiff_P.Title = "Pressure field diff errors over time";
plotFieldDiff_P.Xlabel = "time t";
plotFieldDiff_P.Ylabel = "diff field L2 error norm";

for (int fn = 0; fn < fieldNames.Length; fn++) {

    var fmt = new PlotFormat();
    fmt.Style = Styles.Lines; 
    fmt.DashType = DashTypes.Solid;
    fmt.LineWidth = 2;

    if (fn == 0) {
        fmt.LineColor= LineColors.Blue;
        plotFieldDiff_Vel.AddDataGroup(fieldNames[fn], time, absFieldDiffL2Norm[fn], fmt);
    }
    if (fn == 1) {
        fmt.LineColor= LineColors.Red;
        plotFieldDiff_Vel.AddDataGroup(fieldNames[fn], time, absFieldDiffL2Norm[fn], fmt);
    }
    if (fn == 2) {
        fmt.LineColor= LineColors.Green;
        plotFieldDiff_Vel.AddDataGroup(fieldNames[fn], time, absFieldDiffL2Norm[fn], fmt);
    }
    if (fn == 3) {
        fmt.LineColor= LineColors.Blue;
        plotFieldDiff_P.AddDataGroup(fieldNames[fn], time, absFieldDiffL2Norm[fn], fmt);
    }

}

In [ ]:
plotFieldDiff_Vel.XrangeMin = 0.002;
plotFieldDiff_Vel.XrangeMax = 0.01;
var gpVel = plotFieldDiff_Vel.ToGnuplot();
gpVel.PlotSVG()

Using gnuplot: C:\Users\smuda\AppData\Local\FDY\BoSSS\bin\native\win\gnuplot-gp510-20160418-win32-mingw\gnuplot\bin\gnuplot.exe
Note: In a Jupyter Worksheet, you must NOT have a trailing semicolon in order to see the plot on screen; otherwise, the output migth be surpressed.!


<?xml version="1.0" encoding="utf-8" standalone="no"?>
<!DOCTYPE svg PUBLIC "-//W3C//DTD SVG 1.1//EN"
 "http://www.w3.org/Graphics/SVG/1.1/DTD/svg11.dtd">
 

 Gnuplot 
 Produced by GNUPLOT 5.1 patchlevel 0 

 

 
 

 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 0 
 
 
 
 
 5x10 -12 
 
 
 
 
 1x10 -11 
 
 
 
 
 1.5x10 -11 
 
 
 
 
 2x10 -11 
 
 
 
 
 2.5x10 -11 
 
 
 
 
 3x10 -11 
 
 
 
 
 3.5x10 -11 
 
 
 
 
 4x10 -11 
 
 
 
 
 4.5x10 -11 
 
 
 
 
 0.002 
 
 
 
 
 0.003 
 
 
 
 
 0.004 
 
 
 
 
 0.005 
 
 
 
 
 0.006 
 
 
 
 
 0.007 
 
 
 
 
 0.008 
 
 
 
 
 0.009 
 
 
 
 
 0.01 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 diff field L2 error norm 
 
 
 
 
 time t 
 
 
 
 
 Velocity field diff errors over time 
 
 
 VelocityX 
 
 
 
 
 VelocityX 
 
 
 
	<path stroke='rgb( 0, 0, 255)' d='M577.4,93.1 L630.8,93.1 M113.4,106.6 L121.4,211.8 L129.3,289.7 L137.3,344.9 L145.2,387.0 L153.2,420.9
 L161.2,448.0 L169.1,469.1 L177.1,485.3 L185.0,497.8 L193.0,507.6 L200.9,515.3 L208.9,521.3 L216.9,526.0
 L224.8,529.6 L232.8,532.4 L240.7,534.6 L248.7,536.3 L256.7,537.7 L264.6,538.7 L272.6,539.5 L280.5,540.2
 L288.5,540.7 L296.5,541.0 L304.4,541.3 L312.4,541.6 L320.3,541.8 L328.3,541.9 L336.2,542.0 L344.2,542.1
 L352.2,542.2 L360.1,542.2 L368.1,542.3 L376.0,542.3 L384.0,542.3 L392.0,542.3 L399.9,542.3 L407.9,542.4
 L415.8,542.4 L423.8,542.4 L431.8,542.4 L439.7,542.4 L447.7,542.4 L455.6,542.4 L463.6,542.4 L471.5,542.4
 L479.5,542.4 L487.5,542.4 L495.4,542.4 L503.4,542.4 L511.3,542.4 L519.3,542.4 L527.3,542.4 L535.2,542.4
 L543.2,542.4 L551.1,542.4 L559.1,542.4 L567.0,542.4 L575.0,542.4 L583.0,542.4 L590.9,542.4 L598.9,542.4
 L606.8,542.4 L614.8,542.4 L622.8,542.4 L630.7,542.4 L638.7,542.4 L646.6,542.4 L654.6,542.4 L662.6,542.4
 L670.5,542.4 L678.5,542.4 L686.4,542.4 L694.4,542.4 L702.3,542.4 L710.3,542.4 L718.3,542.4 L726.2,542.4
 L734.2,542.4 L742.1,542.4 L750.1,542.4 '/> 
 
 VelocityY 
 
 
 VelocityY 
 
 
 
	<path stroke='rgb(255, 0, 0)' d='M577.4,117.1 L630.8,117.1 M113.4,469.4 L121.4,486.5 L129.3,500.5 L137.3,509.9 L145.2,516.9 L153.2,523.0
 L161.2,527.4 L169.1,530.7 L177.1,533.3 L185.0,535.3 L193.0,536.8 L200.9,538.1 L208.9,539.0 L216.9,539.8
 L224.8,540.4 L232.8,540.8 L240.7,541.2 L248.7,541.4 L256.7,541.6 L264.6,541.8 L272.6,541.9 L280.5,542.0
 L288.5,542.1 L296.5,542.2 L304.4,542.2 L312.4,542.3 L320.3,542.3 L328.3,542.3 L336.2,542.3 L344.2,542.4
 L352.2,542.4 L360.1,542.4 L368.1,542.4 L376.0,542.4 L384.0,542.4 L392.0,542.4 L399.9,542.4 L407.9,542.4
 L415.8,542.4 L423.8,542.4 L431.8,542.4 L439.7,542.4 L447.7,542.4 L455.6,542.4 L463.6,542.4 L471.5,542.4
 L479.5,542.4 L487.5,542.4 L495.4,542.4 L503.4,542.4 L511.3,542.4 L519.3,542.4 L527.3,542.4 L535.2,542.4
 L543.2,542.4 L551.1,542.4 L559.1,542.4 L567.0,542.4 L575.0,542.4 L583.0,542.4 L590.9,542.4 L598.9,542.4
 L606.8,542.4 L614.8,542.4 L622.8,542.4 L630.7,542.4 L638.7,542.4 L646.6,542.4 L654.6,542.4 L662.6,542.4
 L670.5,542.4 L678.5,542.4 L686.4,542.4 L694.4,542.4 L702.3,542.4 L710.3,542.4 L718.3,542.4 L726.2,542.4
 L734.2,542.4 L742.1,542.4 L750.1,542.4 '/> 
 
 VelocityZ 
 
 
 VelocityZ 
 
 
 
	<path stroke='rgb( 0, 255, 0)' d='M577.4,141.1 L630.8,141.1 M113.4,537.9 L121.4,539.6 L129.3,540.0 L137.3,540.4 L145.2,540.9 L153.2,541.3
 L161.2,541.6 L169.1,541.8 L177.1,541.9 L185.0,542.0 L193.0,542.1 L200.9,542.2 L208.9,542.2 L216.9,542.3
 L224.8,542.3 L232.8,542.3 L240.7,542.3 L248.7,542.3 L256.7,542.4 L264.6,542.4 L272.6,542.4 L280.5,542.4
 L288.5,542.4 L296.5,542.4 L304.4,542.4 L312.4,542.4 L320.3,542.4 L328.3,542.4 L336.2,542.4 L344.2,542.4
 L352.2,542.4 L360.1,542.4 L368.1,542.4 L376.0,542.4 L384.0,542.4 L392.0,542.4 L399.9,542.4 L407.9,542.4
 L415.8,542.4 L423.8,542.4 L431.8,542.4 L439.7,542.4 L447.7,542.4 L455.6,542.4 L463.6,542.4 L471.5,542.4
 L479.5,542.4 L487.5,542.4 L495.4,542.4 L503.4,542.4 L511.3,542.4 L519

In [ ]:
absFieldDiffL2Norm[0][numTs-2]

8.937667344055941E-20

In [ ]:
plotFieldDiff_Vel.SaveToTextFile("results/plotFieldDiff_Vel_" + sess.Name);

In [ ]:
var gpP = plotFieldDiff_P.ToGnuplot();
gpP.PlotSVG()

Using gnuplot: C:\Users\smuda\AppData\Local\FDY\BoSSS\bin\native\win\gnuplot-gp510-20160418-win32-mingw\gnuplot\bin\gnuplot.exe
Note: In a Jupyter Worksheet, you must NOT have a trailing semicolon in order to see the plot on screen; otherwise, the output migth be surpressed.!


<?xml version="1.0" encoding="utf-8" standalone="no"?>
<!DOCTYPE svg PUBLIC "-//W3C//DTD SVG 1.1//EN"
 "http://www.w3.org/Graphics/SVG/1.1/DTD/svg11.dtd">
 

 Gnuplot 
 Produced by GNUPLOT 5.1 patchlevel 0 

 

 
 

 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 0 
 
 
 
 
 0.02 
 
 
 
 
 0.04 
 
 
 
 
 0.06 
 
 
 
 
 0.08 
 
 
 
 
 0.1 
 
 
 
 
 0.12 
 
 
 
 
 0 
 
 
 
 
 0.001 
 
 
 
 
 0.002 
 
 
 
 
 0.003 
 
 
 
 
 0.004 
 
 
 
 
 0.005 
 
 
 
 
 0.006 
 
 
 
 
 0.007 
 
 
 
 
 0.008 
 
 
 
 
 0.009 
 
 
 
 
 0.01 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 diff field L2 error norm 
 
 
 
 
 time t 
 
 
 
 
 Pressure field diff errors over time 
 
 
 Pressure 
 
 
 
 
 Pressure 
 
 
 
	<path stroke='rgb( 0, 0, 255)' d='M588.5,93.1 L641.9,93.1 M86.9,101.2 L93.6,313.8 L100.3,507.2 L107.0,540.1 L113.7,535.3 L120.4,539.7
 L127.1,542.0 L133.8,541.7 L140.5,542.2 L147.2,542.4 L153.9,542.3 L160.6,542.4 L167.3,542.4 L174.0,542.4
 L180.7,542.4 L187.4,542.4 L194.1,542.4 L200.8,542.4 L207.5,542.4 L214.2,542.4 L220.9,542.4 L227.6,542.4
 L234.3,542.4 L241.0,542.4 L247.7,542.4 L254.4,542.4 L261.1,542.4 L267.8,542.4 L274.5,542.4 L281.2,542.4
 L287.9,542.4 L294.6,542.4 L301.3,542.4 L308.0,542.4 L314.7,542.4 L321.4,542.4 L328.1,542.4 L334.8,542.4
 L341.5,542.4 L348.2,542.4 L354.9,542.4 L361.6,542.4 L368.3,542.4 L375.0,542.4 L381.7,542.4 L388.4,542.4
 L395.1,542.4 L401.8,542.4 L408.5,542.4 L415.2,542.4 L421.8,542.4 L428.5,542.4 L435.2,542.4 L441.9,542.4
 L448.6,542.4 L455.3,542.4 L462.0,542.4 L468.7,542.4 L475.4,542.4 L482.1,542.4 L488.8,542.4 L495.5,542.4
 L502.2,542.4 L508.9,542.4 L515.6,542.4 L522.3,542.4 L529.0,542.4 L535.7,542.4 L542.4,542.4 L549.1,542.4
 L555.8,542.4 L562.5,542.4 L569.2,542.4 L575.9,542.4 L582.6,542.4 L589.3,542.4 L596.0,542.4 L602.7,542.4
 L609.4,542.4 L616.1,542.4 L622.8,542.4 L629.5,542.4 L636.2,542.4 L642.9,542.4 L649.6,542.4 L656.3,542.4
 L663.0,542.4 L669.7,542.4 L676.4,542.4 L683.1,542.4 L689.8,542.4 L696.5,542.4 L703.2,542.4 L709.9,542.4
 L716.6,542.4 L723.3,542.4 L730.0,542.4 L736.7,542.4 L743.4,542.4 L750.1,542.4 '/>

In [ ]:
absFieldDiffL2Norm[3][numTs-2]

5.688236842904567E-17

In [ ]:
plotFieldDiff_P.SaveToTextFile("results/plotFieldDiff_P_" + sess.Name);